In [1]:
# 1. Check where Jupyter thinks “.” is and what files live here
import os
print("Notebook working directory:", os.getcwd())
print("Files:", sorted(os.listdir(".")))
# If there’s a subfolder named data, show its contents
if "data" in os.listdir("."):
    print("data/ contains:", sorted(os.listdir("data")))
else:
    print("⚠️ No data/ folder found at this level")

# 2. Now the real imports and path setup
import pandas as pd, numpy as np
from pathlib import Path
from sklearn.base            import BaseEstimator, TransformerMixin
from sklearn.feature_selection import f_classif, mutual_info_classif
from sklearn.pipeline        import Pipeline
from sklearn.compose         import ColumnTransformer
from sklearn.preprocessing   import StandardScaler
from sklearn.model_selection import (GroupKFold, StratifiedKFold,
                                     GridSearchCV, cross_validate)
from sklearn.ensemble        import RandomForestClassifier
from joblib                  import dump

# define where your CSV lives relative to the cwd above
# if the debug print shows you’re in the project root, use data_dir="data"
# if it shows you’re in notebooks/, use data_dir="../data"
data_dir = "../data"  
Path(data_dir).mkdir(exist_ok=True)

Notebook working directory: /home/stewart/KeshenMicroArrayMLApr2025
Files: ['.venv', 'EndoMicroarrayPipeline.ipynb', 'best_logreg.joblib', 'cv_results.json', 'data', 'models', 'plots', 'requirements.txt', 'rf_30gene_leakfree.joblib']
data/ contains: ['GENE_MATRIX_COMBO_MAR31.csv', 'METADATA_COMBO_MAR31_covarsRemoved.csv']


In [2]:
# ------- imports --------
import pandas as pd, numpy as np
from pathlib import Path
from sklearn.base            import BaseEstimator, TransformerMixin
from sklearn.feature_selection import f_classif, mutual_info_classif
from sklearn.pipeline        import Pipeline
from sklearn.compose         import ColumnTransformer
from sklearn.preprocessing   import StandardScaler
from sklearn.model_selection import (GroupKFold, StratifiedKFold,
                                     GridSearchCV, cross_validate)
from sklearn.metrics         import roc_auc_score, roc_curve, confusion_matrix
from joblib                  import dump
import matplotlib.pyplot as plt; import seaborn as sns

# ------- repo paths -----
Path("models").mkdir(exist_ok=True)
Path("plots" ).mkdir(exist_ok=True)

# ------- load data ------
expr = pd.read_csv("data/GENE_MATRIX_COMBO_MAR31.csv", index_col=0).T
meta = pd.read_csv("data/METADATA_COMBO_MAR31_covarsRemoved.csv", index_col=0)
expr = expr.loc[meta.index]

y      = (meta["condition"] == "RIF").astype(int).values
groups = meta["study"].values            # study IDs for Group-CV
print(f"samples={expr.shape[0]}  genes={expr.shape[1]}  classes={np.bincount(y)}")

# ------- leak-safe selector (ANOVA + MI) ----------
class ComboSelector(BaseEstimator, TransformerMixin):
    def __init__(self, k=30): self.k = k
    def fit(self, X, y):
        X_ = X.values if isinstance(X, pd.DataFrame) else X
        f  = f_classif(X_, y)[0]
        mi = mutual_info_classif(X_, y, random_state=0)
        keep = np.argsort(f + mi)[-self.k:]
        self.mask_ = np.zeros(X_.shape[1], dtype=bool); self.mask_[keep] = True
        return self
    def transform(self, X):
        X_ = X.values if isinstance(X, pd.DataFrame) else X
        return X_[:, self.mask_]
    def get_support(self): return self.mask_

numeric_pipe = Pipeline([
    ("select", ComboSelector(k=30)),          # k tuned later
    ("scale",  StandardScaler())
])
preprocess = ColumnTransformer([("genes", numeric_pipe, expr.columns)])


samples=217  genes=10489  classes=[121  96]


In [6]:
# === Cell 3: OOF probabilities → compute AUROC & make plots ===

from sklearn.model_selection import cross_val_predict
from sklearn.metrics         import (roc_auc_score, roc_curve, 
                                     RocCurveDisplay,
                                     confusion_matrix,
                                     ConfusionMatrixDisplay)

probs_all = {}

# 1) get out-of-fold probabilities AND print AUROC
for name, (est, grid) in model_grid.items():
    print(f"\n{name} — fitting + OOF predict…")
    pipe = Pipeline([("prep", preprocess), ("clf", est)])
    gs   = GridSearchCV(pipe, grid, cv=inner,
                        scoring="roc_auc", n_jobs=-1)

    y_proba = cross_val_predict(
        gs, expr, y,
        cv=outer, groups=groups,
        method="predict_proba",
        n_jobs=-1
    )[:, 1]

    auc = roc_auc_score(y, y_proba)
    print(f"  ▶ OOF AUROC = {auc:.3f}")
    probs_all[name] = y_proba

# 2) plot all ROC curves
plt.figure(figsize=(6,6))
for n, p in probs_all.items():
    fpr, tpr, _ = roc_curve(y, p)
    RocCurveDisplay(fpr=fpr, tpr=tpr, roc_auc=roc_auc_score(y, p),
                    estimator_name=n).plot(ax=plt.gca())
plt.plot([0,1],[0,1],"--",color="grey")
plt.title("Leak-free Out-of-Fold ROC Curves")
plt.tight_layout()
plt.savefig("plots/roc_curves.png", dpi=300)
plt.close()

# 3) confusion matrix for the stack
stack_p = probs_all["stack"]
stack_pred = (stack_p >= 0.5).astype(int)
cm = confusion_matrix(y, stack_pred)
ConfusionMatrixDisplay(cm, display_labels=["Control","RIF"]) \
    .plot(cmap="Blues")
plt.title("Stack Confusion Matrix (thr=0.5)")
plt.tight_layout()
plt.savefig("plots/confusion_matrix_stack.png", dpi=300)
plt.close()

# 4) RF feature importance (top-20)
from joblib import load
rf_model = load("models/rf_leakfree.joblib")

mask = ( rf_model.named_steps["prep"]
                   .named_transformers_["genes"]
                   .named_steps["select"]
                   .get_support() )
genes_kept  = expr.columns[mask]
importances = rf_model.named_steps["clf"].feature_importances_

fi = (
    pd.DataFrame({"gene": genes_kept, "importance": importances})
      .sort_values("importance", ascending=False)
      .head(20)
)
plt.figure(figsize=(8,6))
sns.barplot(data=fi, x="importance", y="gene")
plt.title("Top-20 RF Feature Importances")
plt.tight_layout()
plt.savefig("plots/rf_feature_importance.png", dpi=300)
plt.close()

print("\n✅ All plots saved under plots/") 



logreg — fitting + OOF predict…


  ▶ OOF AUROC = 0.538

svm — fitting + OOF predict…
  ▶ OOF AUROC = 0.514

rf — fitting + OOF predict…
  ▶ OOF AUROC = 0.538

stack — fitting + OOF predict…
  ▶ OOF AUROC = 0.524


FileNotFoundError: [Errno 2] No such file or directory: 'models/rf_leakfree.joblib'

In [5]:
# -------- ROC curves
plt.figure(figsize=(6,6))
for n, p in probs_all.items():
    fpr, tpr, _ = roc_curve(y_all, p)
    plt.plot(fpr, tpr, label=f"{n} (AUC {roc_auc_score(y_all,p):.2f})")
plt.plot([0,1],[0,1],'--',c='grey'); plt.legend(); plt.xlabel("FPR"); plt.ylabel("TPR")
plt.title("Out-of-fold ROC curves"); plt.tight_layout()
plt.savefig("plots/roc_curves.png", dpi=300); plt.close()


# -------- confusion matrix for stack
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
stack_pred = (probs_all["stack"] >= 0.5).astype(int)
cm = confusion_matrix(y_all, stack_pred)
ConfusionMatrixDisplay(cm, display_labels=["Control","RIF"]).plot(cmap="Blues")
plt.title("Stack confusion matrix (thr=0.5)"); plt.tight_layout()
plt.savefig("plots/confusion_matrix_stack.png", dpi=300); plt.close()

# -------- RF feature importance (top 20)
rf_model = dump.load("models/rf_leakfree.joblib")
genes = expr.columns[rf_model.named_steps["prep"]
                         .named_transformers_["genes"]
                         .named_steps["select"].get_support()]
imp = rf_model.named_steps["clf"].feature_importances_
fi = pd.DataFrame({"gene":genes, "imp":imp}).sort_values("imp", ascending=False).head(20)
sns.barplot(data=fi, x="imp", y="gene"); plt.title("Top-20 RF feature importance")
plt.tight_layout(); plt.savefig("plots/rf_feature_importance.png", dpi=300); plt.close()


InvalidParameterError: The 'y_true' parameter of roc_curve must be an array-like. Got None instead.

<Figure size 600x600 with 0 Axes>

In [ ]:
best = dump.load("models/stack_leakfree.joblib")
# (for RF: load models/rf_leakfree.joblib)
print(best)
